In [1]:
# Cell 1 — Imports, settings, and loading data + base model

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler

import joblib
import json
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Libraries imported.")

# Paths
processed_path = "../data/processed"
models_path    = "../models"

# Load datasets (with lifestyle + cluster + demographics + Stress_Level)
train_df = pd.read_csv(os.path.join(processed_path, "train_with_clusters.csv"))
val_df   = pd.read_csv(os.path.join(processed_path, "val_with_clusters.csv"))
test_df  = pd.read_csv(os.path.join(processed_path, "test_with_clusters.csv"))

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

# Load base model: Lifestyle + Cluster (Model B from Notebook 04)
base_model_path = os.path.join(models_path, "multinomial_logreg_lifestyle_cluster.pkl")
model_base = joblib.load(base_model_path)

# Load feature config from Notebook 04
config_path = os.path.join(models_path, "model_config.json")
with open(config_path, "r") as f:
    config = json.load(f)

features_base = config["features_model_B"]   # lifestyle + cluster
target_col    = config["target"]
class_labels  = config["classes"]

print("\nBase model and config loaded.")
print("Base model features:", features_base)
print("Target column:", target_col)
print("Class labels:", class_labels)


Libraries imported.
Train shape: (7868, 13)
Val shape:   (1681, 13)
Test shape:  (1686, 13)

Base model and config loaded.
Base model features: ['Sleep_Hours_Scaled', 'Work_Hours_Scaled', 'Physical_Activity_Hours_Scaled', 'Social_Media_Usage_Scaled', 'Diet_Quality_Encoded_Scaled', 'Smoking_Habit_Encoded_Scaled', 'Alcohol_Consumption_Encoded_Scaled', 'Cluster']
Target column: Stress_Level
Class labels: ['High', 'Low', 'Medium']


In [2]:
# Cell 2 — Define column groups (lifestyle, cluster, demographics)

# Lifestyle features: same logic as before (scaled habits only)
lifestyle_features = [
    col for col in train_df.columns
    if col.endswith("_Scaled") and not col.startswith("Age")
]

cluster_feature = ["Cluster"]

print("Lifestyle features:", lifestyle_features)
print("Cluster feature:", cluster_feature)

# Demographic raw columns: Age / Age_Scaled, Gender, Occupation, Country
demo_raw = []

if "Age_Scaled" in train_df.columns:
    age_col = "Age_Scaled"
    demo_raw.append(age_col)
elif "Age" in train_df.columns:
    age_col = "Age"
    demo_raw.append(age_col)
else:
    age_col = None
    print("WARNING: No Age or Age_Scaled column found.")

for col in ["Gender", "Occupation", "Country"]:
    if col in train_df.columns:
        demo_raw.append(col)

print("Demographic raw columns:", demo_raw)
print("Unique Stress Levels in training set:", train_df[target_col].unique())


Lifestyle features: ['Sleep_Hours_Scaled', 'Work_Hours_Scaled', 'Physical_Activity_Hours_Scaled', 'Social_Media_Usage_Scaled', 'Diet_Quality_Encoded_Scaled', 'Smoking_Habit_Encoded_Scaled', 'Alcohol_Consumption_Encoded_Scaled']
Cluster feature: ['Cluster']
Demographic raw columns: ['Age', 'Gender', 'Occupation', 'Country']
Unique Stress Levels in training set: ['Medium' 'Low' 'High']


In [3]:
# Cell 3 — Preprocess demographic variables:
# - Scale Age 
# - One-hot encode Gender, Occupation, Country
# - Align val/test dummy columns with train

train_demo = train_df[demo_raw].copy()
val_demo   = val_df[demo_raw].copy()
test_demo  = test_df[demo_raw].copy()

# 1) Handle Age scaling
if age_col == "Age":
    scaler_age = MinMaxScaler()
    train_demo["Age_Scaled"] = scaler_age.fit_transform(train_demo[["Age"]])
    val_demo["Age_Scaled"]   = scaler_age.transform(val_demo[["Age"]])
    test_demo["Age_Scaled"]  = scaler_age.transform(test_demo[["Age"]])
elif age_col == "Age_Scaled":
    # already scaled; nothing to do
    pass

# From this point, we will use Age_Scaled (if available)
if "Age_Scaled" in train_demo.columns:
    age_feature = "Age_Scaled"
else:
    age_feature = None
    print("No Age_Scaled available after preprocessing.")

# 2) One-hot encode Gender, Occupation, Country
cat_cols = [c for c in ["Gender", "Occupation", "Country"] if c in train_demo.columns]

train_cats = pd.get_dummies(train_demo[cat_cols], drop_first=False)
val_cats   = pd.get_dummies(val_demo[cat_cols], drop_first=False)
test_cats  = pd.get_dummies(test_demo[cat_cols], drop_first=False)

# Align val/test categorical columns with train (fill missing with 0)
val_cats  = val_cats.reindex(columns=train_cats.columns, fill_value=0)
test_cats = test_cats.reindex(columns=train_cats.columns, fill_value=0)

# 3) Combine Age_Scaled (if present) + categorical dummies into processed demographic matrices
if age_feature is not None:
    demo_train_processed = pd.concat([train_demo[[age_feature]], train_cats], axis=1)
    demo_val_processed   = pd.concat([val_demo[[age_feature]],   val_cats],   axis=1)
    demo_test_processed  = pd.concat([test_demo[[age_feature]],  test_cats],  axis=1)
else:
    demo_train_processed = train_cats.copy()
    demo_val_processed   = val_cats.copy()
    demo_test_processed  = test_cats.copy()

print("Processed demographic feature shapes:")
print("  Train demo:", demo_train_processed.shape)
print("  Val demo:  ", demo_val_processed.shape)
print("  Test demo: ", demo_test_processed.shape)

demo_train_processed.head()


Processed demographic feature shapes:
  Train demo: (7868, 19)
  Val demo:   (1681, 19)
  Test demo:  (1686, 19)


,Age_Scaled,Gender_Female,Gender_Male,Gender_Non-binary,Gender_Prefer not to say,Occupation_Education,Occupation_Engineering,Occupation_Finance,Occupation_Healthcare,Occupation_IT,Occupation_Other,Occupation_Sales,Country_Australia,Country_Canada,Country_Germany,Country_India,Country_Other,Country_UK,Country_USA
0,0.063830,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False
1,0.127660,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
2,0.723404,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False
3,0.361702,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False
4,0.510638,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False


In [4]:
# Cell 4 — Build X and y for base (loaded) and demo (new) models

# Target (same for both models)
y_train = train_df[target_col]
y_val   = val_df[target_col]
y_test  = test_df[target_col]

# Base model features — EXACTLY as used in Notebook 04
X_train_base = train_df[features_base]
X_val_base   = val_df[features_base]
X_test_base  = test_df[features_base]

print("Base model feature shape (train):", X_train_base.shape)

# Demo model features: base features + demographics
X_train_demo = pd.concat([X_train_base.reset_index(drop=True),
                          demo_train_processed.reset_index(drop=True)], axis=1)
X_val_demo   = pd.concat([X_val_base.reset_index(drop=True),
                          demo_val_processed.reset_index(drop=True)],   axis=1)
X_test_demo  = pd.concat([X_test_base.reset_index(drop=True),
                          demo_test_processed.reset_index(drop=True)],  axis=1)

print("Demographic model feature shape (train):", X_train_demo.shape)
X_train_demo.head()


Base model feature shape (train): (7868, 8)
Demographic model feature shape (train): (7868, 27)


,Sleep_Hours_Scaled,Work_Hours_Scaled,Physical_Activity_Hours_Scaled,Social_Media_Usage_Scaled,Diet_Quality_Encoded_Scaled,Smoking_Habit_Encoded_Scaled,Alcohol_Consumption_Encoded_Scaled,Cluster,Age_Scaled,Gender_Female,Gender_Male,Gender_Non-binary,Gender_Prefer not to say,Occupation_Education,Occupation_Engineering,Occupation_Finance,Occupation_Healthcare,Occupation_IT,Occupation_Other,Occupation_Sales,Country_Australia,Country_Canada,Country_Germany,Country_India,Country_Other,Country_UK,Country_USA
0,0.133333,0.14,0.2,0.527273,0.0,1.000000,0.000000,0,0.063830,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False
1,0.916667,0.68,0.5,0.563636,0.0,0.000000,0.333333,0,0.127660,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True
2,0.183333,0.60,0.3,0.090909,1.0,0.666667,1.000000,2,0.723404,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False
3,0.633333,0.92,0.8,0.145455,0.5,0.000000,0.000000,0,0.361702,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False
4,0.416667,0.96,0.8,0.781818,0.5,0.333333,1.000000,1,0.510638,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False


In [5]:
# Cell 5 — Evaluate the already-trained BASE model (lifestyle + cluster)

# Validation performance
val_pred_base = model_base.predict(X_val_base)
val_acc_base  = accuracy_score(y_val, val_pred_base)

print("===== BASE MODEL (Lifestyle + Cluster) — VALIDATION =====")
print("Validation Accuracy:", round(val_acc_base, 4))
print(classification_report(y_val, val_pred_base, digits=3))

# Test performance
test_pred_base = model_base.predict(X_test_base)
test_acc_base  = accuracy_score(y_test, test_pred_base)

print("\n===== BASE MODEL (Lifestyle + Cluster) — TEST =====")
print("Test Accuracy:", round(test_acc_base, 4))
print(classification_report(y_test, test_pred_base, digits=3))


===== BASE MODEL (Lifestyle + Cluster) — VALIDATION =====
Validation Accuracy: 0.9334
              precision    recall  f1-score   support

        High      0.958     0.957     0.957       806
         Low      0.950     0.953     0.952       443
      Medium      0.870     0.870     0.870       432

    accuracy                          0.933      1681
   macro avg      0.926     0.927     0.926      1681
weighted avg      0.933     0.933     0.933      1681


===== BASE MODEL (Lifestyle + Cluster) — TEST =====
Test Accuracy: 0.9437
              precision    recall  f1-score   support

        High      0.964     0.958     0.961       809
         Low      0.975     0.953     0.964       444
      Medium      0.877     0.908     0.892       433

    accuracy                          0.944      1686
   macro avg      0.939     0.939     0.939      1686
weighted avg      0.944     0.944     0.944      1686



In [6]:
# Cell 6 — Train DEMOGRAPHIC model (Lifestyle + Cluster + Demographics)

model_demo = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

model_demo.fit(X_train_demo, y_train)

print("Demographic model trained (lifestyle + cluster + demographics).")


Demographic model trained (lifestyle + cluster + demographics).


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [7]:
# Cell 7 — Validation performance comparison: Base vs Demo model

val_pred_demo = model_demo.predict(X_val_demo)
val_acc_demo  = accuracy_score(y_val, val_pred_demo)

print("===== VALIDATION ACCURACY COMPARISON =====")
print(f"Base model (Lifestyle + Cluster):                {val_acc_base:.4f}")
print(f"Demographic model (Lifestyle + Cluster + Demo):  {val_acc_demo:.4f}")

print("\n--- Base model (Validation) ---")
print(classification_report(y_val, val_pred_base, digits=3))

print("\n--- Demo model (Validation) ---")
print(classification_report(y_val, val_pred_demo, digits=3))


===== VALIDATION ACCURACY COMPARISON =====
Base model (Lifestyle + Cluster):                0.9334
Demographic model (Lifestyle + Cluster + Demo):  0.9328

--- Base model (Validation) ---
              precision    recall  f1-score   support

        High      0.958     0.957     0.957       806
         Low      0.950     0.953     0.952       443
      Medium      0.870     0.870     0.870       432

    accuracy                          0.933      1681
   macro avg      0.926     0.927     0.926      1681
weighted avg      0.933     0.933     0.933      1681


--- Demo model (Validation) ---
              precision    recall  f1-score   support

        High      0.960     0.955     0.958       806
         Low      0.942     0.957     0.950       443
      Medium      0.872     0.866     0.869       432

    accuracy                          0.933      1681
   macro avg      0.925     0.926     0.925      1681
weighted avg      0.933     0.933     0.933      1681



In [8]:
# Cell 8 — Test performance comparison: Base vs Demo model

test_pred_demo = model_demo.predict(X_test_demo)
test_acc_demo  = accuracy_score(y_test, test_pred_demo)

print("===== TEST ACCURACY COMPARISON =====")
print(f"Base model (Lifestyle + Cluster):                {test_acc_base:.4f}")
print(f"Demographic model (Lifestyle + Cluster + Demo):  {test_acc_demo:.4f}")

print("\n--- Base model (Test) ---")
print(classification_report(y_test, test_pred_base, digits=3))

print("\n--- Demo model (Test) ---")
print(classification_report(y_test, test_pred_demo, digits=3))


===== TEST ACCURACY COMPARISON =====
Base model (Lifestyle + Cluster):                0.9437
Demographic model (Lifestyle + Cluster + Demo):  0.9395

--- Base model (Test) ---
              precision    recall  f1-score   support

        High      0.964     0.958     0.961       809
         Low      0.975     0.953     0.964       444
      Medium      0.877     0.908     0.892       433

    accuracy                          0.944      1686
   macro avg      0.939     0.939     0.939      1686
weighted avg      0.944     0.944     0.944      1686


--- Demo model (Test) ---
              precision    recall  f1-score   support

        High      0.959     0.957     0.958       809
         Low      0.972     0.950     0.961       444
      Medium      0.872     0.896     0.884       433

    accuracy                          0.940      1686
   macro avg      0.934     0.934     0.934      1686
weighted avg      0.940     0.940     0.940      1686



In [9]:
# Cell 9 — Inspect coefficients of demographic features in the Demo model

demo_feature_names = list(demo_train_processed.columns)

# Get full coefficient matrix: rows = classes, cols = all features in X_train_demo
coef_df = pd.DataFrame(
    model_demo.coef_,
    columns=X_train_demo.columns,
    index=model_demo.classes_
)

demo_coefs_only = coef_df[demo_feature_names]

print("===== Demographic Coefficients per Class =====")
display(demo_coefs_only.round(4))


===== Demographic Coefficients per Class =====


,Age_Scaled,Gender_Female,Gender_Male,Gender_Non-binary,Gender_Prefer not to say,Occupation_Education,Occupation_Engineering,Occupation_Finance,Occupation_Healthcare,Occupation_IT,Occupation_Other,Occupation_Sales,Country_Australia,Country_Canada,Country_Germany,Country_India,Country_Other,Country_UK,Country_USA
High,-0.1458,-0.0253,0.0549,0.0362,-0.0350,0.0473,-0.0529,0.0202,0.0632,-0.1077,0.1167,-0.0561,-0.1742,0.0802,0.0281,0.0948,-0.0231,-0.0067,0.0317
Low,0.1017,-0.0363,-0.0014,-0.0069,0.0221,-0.0398,-0.0568,-0.0661,0.0322,0.1477,-0.1294,0.0896,0.1603,-0.0468,-0.0773,-0.0208,0.0472,0.0150,-0.1002
Medium,0.0441,0.0616,-0.0535,-0.0292,0.0130,-0.0076,0.1097,0.0459,-0.0954,-0.0399,0.0127,-0.0336,0.0139,-0.0333,0.0493,-0.0740,-0.0241,-0.0083,0.0685
